# 00 Smoke test: Medusae Fossae x Egypt (real data, terrain only)

Goal: one real end-to-end run. MOLA target tile + ETOPO Egypt tiles -> `run_real.py` -> `out/`.
No climate features yet, so only the terrain bars are scored. **Not a finding until validated.**

Raw downloads and results are kept on Google Drive, so a new Colab session skips the downloads.
Clear all outputs before committing this notebook (Edit -> Clear all outputs).

In [ ]:
# 1. Repo + install (re-run after every new session)
import os
if not os.path.exists('/content/earth-twin'):
    !git clone -q https://github.com/Demerdashh/earth-twin.git
%cd /content/earth-twin
!git pull -q
!pip -q install -e ".[real]"
!python tests/test_core.py

In [ ]:
# 2. Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/earth-twin')
RAW, DATA = DRIVE / 'raw', DRIVE / 'data'
for p in (RAW, DATA, DATA / 'targets'):
    p.mkdir(parents=True, exist_ok=True)
print(DRIVE)

## Mars target: MOLA MEGDR 128 px/deg, tile megt44n180hb

Covers 44N to 0 and 180E to 270E (PDS label megt44n180hb.xml): 5632 lines x 11520 samples,
SignedMSB2 (big-endian int16), metres, equirectangular, 128 px/deg. The target (11N, 159W = 201E) is inside.
We read the raw file with numpy instead of GDAL, so there are no PDS label issues.

In [ ]:
# 3. Download the MOLA tile (about 130 MB, once)
MOLA = RAW / 'megt44n180hb.img'
URL = 'https://pds-geosciences.wustl.edu/mgs/urn-nasa-pds-mgs_mola_topography_derived/meg128/megt44n180hb.img'
if not MOLA.exists() or MOLA.stat().st_size != 129_761_280:
    !wget -q -c -O "{MOLA}" "{URL}"
print(MOLA, MOLA.stat().st_size, 'bytes (expected 129761280)')

In [ ]:
# 4. Cut the Medusae Fossae target tile at 463 m x 64 px (same format as scripts/cut_target.py)
import json, numpy as np
from IPython.display import Image, display
from analoglens.geo import GlobalRaster
from analoglens.render import save_relief_png

LINES, SAMPLES = 5632, 11520
raw = np.fromfile(MOLA, dtype='>i2')
assert raw.size == LINES * SAMPLES, raw.size
mola = GlobalRaster(raw.reshape(LINES, SAMPLES), lat_top=44.0, lon_left=180.0, deg_per_px=1 / 128)
del raw

KEY = 'medusae_fossae'
cfg = json.loads(Path('config/targets.json').read_text(encoding='utf-8'))[KEY]
lat, lon = cfg['lat'], cfg['lon']          # change here to try another spot inside the formation
tile = mola.tile(lat, lon, 64, 463.0, 'mars')
print(f'valid {np.isfinite(tile).mean():.0%}, relief {np.nanmax(tile) - np.nanmin(tile):.0f} m')
np.savez_compressed(DATA / 'targets' / f'{KEY}.npz', tile=tile.astype(np.float32), key=KEY, body='mars',
                    lat=lat, lon=lon, res_m=463.0)
save_relief_png(tile, 463.0, DATA / 'targets' / f'{KEY}.png')
display(Image(str(DATA / 'targets' / f'{KEY}.png'), width=320))

**Stop and look at the PNG.** Linear, wind-aligned ridges = good. Smooth or blotchy = the
463 m grid is too coarse or interpolated here; move `lat`/`lon` inside the formation and re-run cell 4.

## If the preview looks smooth: scan nearby spots

Run 4b, pick a row from the table, put its numbers into cell 4 (`lat, lon = 12.0, -157.0`), re-run cell 4.
Longitude is negative for west: 159W = -159 (= 201E).

In [ ]:
# 4b. Scan around the target: which nearby spot shows the most linear terrain at 463 m?
#     Needs `mola` from cell 4. Prints the 10 most linear tiles and shows them side by side.
import numpy as np
from PIL import Image as PILImage
from IPython.display import display
from analoglens.terrain import anisotropy, detrend, fill_nans, gradients
from analoglens.render import relief_rgb

def ridge_azimuth(dem, res_m):
    # ridge direction in degrees from north (0 = N-S, 90 = E-W, 45 = NE-SW, 135 = NW-SE)
    z = detrend(fill_nans(dem)); gx, gy = gradients(z, res_m)
    jxx, jyy, jxy = (gx * gx).mean(), (gy * gy).mean(), (gx * gy).mean()
    return (-0.5 * np.degrees(np.arctan2(2 * jxy, jxx - jyy))) % 180

rows = []
for la in np.arange(8.0, 14.01, 1.0):          # 7 x 7 points, 1 degree (~59 km) apart
    for lo in np.arange(-162.0, -155.99, 1.0):
        t = mola.tile(la, lo, 64, 463.0, 'mars')
        rows.append(dict(lat=la, lon=lo, aniso=anisotropy(fill_nans(t)), az=ridge_azimuth(t, 463.0),
                         relief=float(np.nanmax(t) - np.nanmin(t)), tile=t))
rows.sort(key=lambda r: -r['aniso'])
print(' lat     lon    linearity  ridge az  relief')
for r in rows[:10]:
    print(f"{r['lat']:5.1f} {r['lon']:7.1f}   {r['aniso']:.2f}      {r['az']:5.0f}°   {r['relief']:6.0f} m")
lin = [r for r in rows if r['aniso'] >= 0.3]
ns = sum(min(r['az'], 180 - r['az']) < 15 for r in lin)
print(f"\n{ns} of {len(lin)} clearly linear tiles (linearity >= 0.3) run within 15° of N-S.")
print("MOLA tracks run roughly N-S near the equator, so if nearly all are N-S, suspect track artifacts.")
sheet = np.concatenate([relief_rgb(r['tile'], 463.0) for r in rows[:5]], axis=1)
display(PILImage.fromarray(sheet).resize((5 * 192, 192)))

In [ ]:
# 5. Download ETOPO 2022 15 arc-second surface tiles for Egypt (about 30 MB each, once)
BASE = 'https://www.ngdc.noaa.gov/mgg/global/relief/ETOPO2022/data/15s/15s_surface_elev_gtif'
ETOPO = []
for name in ['ETOPO_2022_v1_15s_N30E015_surface.tif', 'ETOPO_2022_v1_15s_N30E030_surface.tif']:
    path = RAW / name
    if not path.exists():
        !wget -q -c -O "{path}" "{BASE}/{name}"
    ETOPO.append(path)
    print(path.name, path.stat().st_size // 2**20, 'MB')

In [ ]:
# 5b. Two more ETOPO tiles for known analog sites (Lut, Death Valley, Meteor Crater, MDRS, Craters of the Moon)
KNOWN = []
for name in ['ETOPO_2022_v1_15s_N45E045_surface.tif', 'ETOPO_2022_v1_15s_N45W120_surface.tif']:
    path = RAW / name
    if not path.exists() or path.stat().st_size < 1_000_000:
        !wget -q -c -O "{path}" "{BASE}/{name}"
    KNOWN.append(path)
    print(path.name, path.stat().st_size // 2**20, 'MB')   # 0 MB = wrong file name, check the NOAA listing

In [ ]:
# 6. Earth tiles: NE Africa + W Arabia, 15-30N 15-45E (same two rasters), and the known sites
!python scripts/cut_tiles.py --raster "{ETOPO[0]}" --raster "{ETOPO[1]}" \
    --bbox 15 30 15 45 --step-km 30 --prefix NE --out "{DATA}/earth_ne.npz"
!python scripts/cut_tiles.py --raster "{KNOWN[0]}" --raster "{KNOWN[1]}" \
    --centers config/known_analogs.json --prefix KN --out "{DATA}/earth_known.npz"

In [ ]:
# 7. Rank with both embedders. Add the two --features lines (see FEATURES) once NDVI is ready.
import os
FEATURES = ''
if (DATA / 'earth_ne_features.npz').exists() and (DATA / 'earth_known_features.npz').exists():
    FEATURES = f'--features "{DATA}/earth_ne_features.npz" --features "{DATA}/earth_known_features.npz" --max-ndvi 0.3'
print('features:', FEATURES or 'none (terrain only)')
for emb in ('descriptors', 'dino'):
    cmd = (f'python scripts/run_real.py --earth "{DATA}/earth_ne.npz" --earth "{DATA}/earth_known.npz" '
           f'--targets "{DATA}/targets" --embedder {emb} --cards 50 {FEATURES} --out /content/out_{emb[:4]}')
    !{cmd}

In [ ]:
# 8. Known-site percentiles and the top 10, per task and embedder
import json
def show(folder):
    val = json.loads(Path(f'{folder}/validation.json').read_text())
    print(f"\n===== {folder} | embedder {val['embedder']} | synthetic {val['synthetic']}")
    for key, v in val['results'].items():
        print(f"-- {key}: AUC {v['auc']} (n_positive {v['n_positive']})")
        for s in v['known_sites']:
            print('    ', s)
for name in ('out_desc', 'out_dino'):
    show(f'/content/{name}')
    r = json.loads(Path(f'/content/{name}/rank/medusae_fossae__wind_erosion_science.json').read_text())
    for x in r['results'][:10]:
        print(f"  {x['rank']:>2} {x['tile_id']} {x['lat']:7.3f} {x['lon']:7.3f} score {x['score']:.3f} novel {x['novel']} {x['dims']}")

In [ ]:
# 8b. Where do the Egypt deep-dive sites rank? (best tile within 30 km of each site)
!python scripts/run_real.py --earth "{DATA}/earth_ne.npz" --earth "{DATA}/earth_known.npz" \
    --targets "{DATA}/targets" --embedder dino --cards 1 {FEATURES} \
    --known config/egypt_candidates.json --radius-km 30 --out /content/out_egypt_check
show('/content/out_egypt_check')

In [ ]:
# 9. Save to Drive: out_dino is the one the website uses (copy its contents into the site's data/ folder)
import shutil
for name in ('out_desc', 'out_dino', 'out_egypt_check'):
    shutil.make_archive(str(DRIVE / name), 'zip', f'/content/{name}')
!ls -la "{DRIVE}"